# Does news propagate through a company graph?

When news hits one company, can we predict how the companies linked to it move? We predict whether
each NASDAQ-100 member closes up or down 1 and 5 trading days ahead (2019–2023). Models are scored on
**within-day ranking**: on each day, are the risers ranked above the fallers?

Every number below is read from the committed files in `results/`; nothing is retrained or
recomputed. With the team's data folder (`NDX_DATA_DIR`), a few cells also show the raw artifacts.
Without it, they say what is missing and show the committed tables instead.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
REPO = next((p for p in (cwd, *cwd.parents) if (p / "results/tables").is_dir()), None)
if REPO is None:
    raise RuntimeError("Open the notebook from inside the nasdaq100-news-graph checkout")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
TABLES, ATTN = REPO / "results/tables", REPO / "results/attention"
pd.set_option("display.max_colwidth", 60)

try:
    from src.core.paths import data_dir
    DATA = data_dir()
except Exception:
    DATA = REPO / "data"


def data_file(rel):
    """The artifact's path if present; otherwise None, after saying what is missing."""
    p = DATA / rel
    if p.is_file():
        return p
    print(f"Missing {rel} in the data folder: showing the committed tables in results/ instead.")
    return None


def _after(lines, markers):
    i = 0
    for m in markers:
        i = next(j for j in range(i, len(lines)) if m in lines[j]) + 1
    return i


def md_table(path, *markers, nth=0):
    """The nth Markdown table after the lines containing each marker in turn, cells as written."""
    lines = (REPO / path).read_text(encoding="utf-8").splitlines()
    i = _after(lines, markers)
    for _ in range(nth + 1):
        while not lines[i].startswith("|"):
            i += 1
        start = i
        while i < len(lines) and lines[i].startswith("|"):
            i += 1
    rows = [[c.strip().replace("**", "") for c in ln.strip().strip("|").split("|")] for ln in lines[start:i]]
    return pd.DataFrame(rows[2:], columns=rows[0])


def md_bullets(path, *markers):
    """The bullet list after the markers, as written."""
    lines = (REPO / path).read_text(encoding="utf-8").splitlines()
    i = _after(lines, markers)
    while not lines[i].startswith("- "):
        i += 1
    out = []
    while i < len(lines) and lines[i].startswith("- "):
        out.append(lines[i])
        i += 1
    return Markdown("\n".join(out))


print(f"Team data folder (NDX_DATA_DIR): {'present' if (DATA / 'prices.parquet').is_file() else 'absent'}")

Team data folder (NDX_DATA_DIR): present


## 1 · The data

Daily prices for every company that was in the index at some point in 2019–2023. Membership is
rebuilt from the historical record, so a company counts only on days it was a member. Headlines come
from FNSPID and are scored with FinBERT, then averaged per company-day. Coverage is sparse: about half
of company-days have no article, and `has_news = 0` keeps "no coverage" apart from neutral news.

In [2]:
coverage = pd.read_csv(TABLES / "news_coverage_by_year.csv")
total = coverage[["ticker_days", "with_news"]].sum()
print(f"In-index company-days 2019-2023: {total.ticker_days:,}; with news that day: "
      f"{total.with_news / total.ticker_days:.1%}  (results/tables/news_coverage_by_year.csv)")

prices, news = data_file("prices.parquet"), data_file("news_features.parquet")
if prices is not None and news is not None:
    p = pd.read_parquet(prices, columns=["date", "ticker", "in_index"])
    window = p[(p["date"] >= "2019-01-01") & (p["date"] <= "2023-12-31")]
    print(f"prices.parquet: {len(p):,} rows, {p['ticker'].nunique()} tickers, {p['date'].min().date()} to "
          f"{p['date'].max().date()}; {window['date'].nunique():,} trading days in 2019-2023")
    preview = pd.read_parquet(news, columns=["date", "ticker", "has_news", "n_articles", "sentiment", "confidence"])
    print("news_features.parquet preview (one row per in-index company-day; 768 embedding columns not shown):")
    display(preview[preview["date"] == "2023-05-17"].sort_values("n_articles", ascending=False).head(6))
else:
    display(coverage[["year", "ticker_days", "with_news", "share_with_news", "tickers", "tickers_without_news"]])

In-index company-days 2019-2023: 128,519; with news that day: 49.6%  (results/tables/news_coverage_by_year.csv)
prices.parquet: 238,195 rows, 174 tickers, 2018-01-02 to 2023-12-29; 1,258 trading days in 2019-2023
news_features.parquet preview (one row per in-index company-day; 768 embedding columns not shown):


,date,ticker,has_news,n_articles,sentiment,confidence
112730,2023-05-17,MSFT,True,26,0.035283,0.811240
112753,2023-05-17,TSLA,True,19,0.146103,0.848921
112707,2023-05-17,GOOG,True,17,0.149503,0.835756
112672,2023-05-17,AMGN,True,17,-0.673289,0.912658
112662,2023-05-17,AAPL,True,17,0.141569,0.855032
112708,2023-05-17,GOOGL,True,17,0.149503,0.835756


## 2 · The graph

Three edge types link companies:
- **`sector`:** same GICS industry. Static.
- **`comention`:** Jaccard overlap of the two firms' articles.
- **`corr`:** partial correlation of daily returns, net of the market.

`comention` and `corr` are rebuilt for each fold from its training window only, and each ticker keeps
its 10 strongest peers.

In [3]:
com = md_table("results/tables/comention_report.md", "SCHEMA.md defaults")
cor = md_table("results/tables/corr_report.md", "Density after top-k", "**residual**")
counts = pd.DataFrame({
    "fold": com["fold"],
    "comention pairs": com["edges"], "comention mean degree": com["mean degree"],
    "corr pairs": cor["edges"], "corr mean degree": cor["mean degree"],
}).set_index("fold")
display(counts)

edges = data_file("edges.parquet")
if edges is not None:
    e = pd.read_parquet(edges)
    s = e[e["edge_type"] == "sector"]
    print(f"sector (every fold): {len(s) // 2} pairs over {s['dst'].nunique()} companies, "
          f"mean degree {s.groupby('dst').size().mean():.1f}")
else:
    print("sector (every fold): 940 pairs over 169 companies, mean degree 11.1 (report Section 4, Method)")

pairs = pd.read_csv(TABLES / "comention_top_pairs.csv")
top = pairs[pairs["fold"] == 4].sort_values("weight", ascending=False).head(12).reset_index(drop=True)
print("\nStrongest co-mention edges, fold 4 (training window 2021-2022):")
display(top[["src", "dst", "weight"]].rename(columns={"src": "a", "dst": "b", "weight": "Jaccard"}))

corr = pd.read_csv(TABLES / "corr_top_pairs.csv")
deals = {("ADI", "MXIM"), ("ALXN", "AZN"), ("AMD", "XLNX")}
hits = corr[(corr["fold"] == 4) & corr.apply(lambda r: (r["src"], r["dst"]) in deals, axis=1)]
days = md_table("results/tables/corr_report.md", "Top 15 pairs, fold 4", "**residual**").set_index(["a", "b"])["n_obs"]
print("Stock-for-stock acquisitions among the strongest corr edges, fold 4:")
display(hits.assign(shared_days=[days[(a, b)] for a, b in zip(hits["src"], hits["dst"], strict=True)])[
    ["src", "dst", "weight", "shared_days"]].rename(columns={"src": "a", "dst": "b", "weight": "partial corr"})
    .reset_index(drop=True))
if edges is not None:
    atvi = e[(e["src"] == "MSFT") & (e["dst"] == "ATVI")]
    print("MSFT-ATVI (all-cash deal, announced January 2022) edges by fold:")
    display(atvi.pivot_table(index="fold", columns="edge_type", values="weight").reindex(range(5)).round(3))
else:
    print("MSFT-ATVI: no corr edge in folds 3-4 (partial correlation -0.03 in fold 4, Method section)")

,comention pairs,comention mean degree,corr pairs,corr mean degree
fold,,,,
0,920,12.9,858,14.1
1,914,12.8,843,13.9
2,908,12.7,863,14.0
3,734,10.9,841,13.8
4,756,11.2,831,13.8


sector (every fold): 940 pairs over 169 companies, mean degree 11.1

Strongest co-mention edges, fold 4 (training window 2021-2022):


,a,b,Jaccard
0,EA,TTWO,0.1356
1,AAPL,MSFT,0.1160
2,AMD,INTC,0.1105
3,AMD,NVDA,0.1010
4,BKNG,EXPE,0.0995
5,CRWD,PANW,0.0618
6,CMCSA,NFLX,0.0563
7,AAL,UAL,0.0544
8,ABNB,BKNG,0.0504
9,GOOG,MSFT,0.0499


Stock-for-stock acquisitions among the strongest corr edges, fold 4:


,a,b,partial corr,shared_days
0,ADI,MXIM,0.8817,164
1,ALXN,AZN,0.8722,138
2,AMD,XLNX,0.7192,282


MSFT-ATVI (all-cash deal, announced January 2022) edges by fold:


edge_type,comention,corr
fold,,
0,NaN,0.245
1,NaN,0.242
2,NaN,NaN
3,0.018,NaN
4,0.025,NaN


**Reading.**
- **Co-mention pairs.** EA–TTWO, BKNG–EXPE, CRWD–PANW, AAL–UAL and EBAY–PYPL are genuine
  competitors. No list of peers was supplied: the Jaccard weight surfaced them from the headlines alone.
- **Acquisitions.** The `corr` edges picked up three stock-for-stock acquisitions while each was
  pending: ADI–MXIM, ALXN–AZN and AMD–XLNX. The all-cash MSFT–ATVI deal is the opposite case.
  - **Before the deal** (folds 0–1, training windows ending before January 2022), the two were
    ordinary `corr` peers.
  - **While it was pending** (folds 3–4), they are not: partial correlation −0.03 in fold 4 (Method
    section). The pair is then linked only by co-mention, through the deal news.
- **Why that is right.** A share-funded target trades as a fixed ratio of its acquirer, so their
  prices are tied. A cash offer pins the target to a fixed price instead. The edges follow how prices
  are actually linked, not a label saying a relationship exists. Removing these three pairs changes
  within-day AUC by at most 0.0006 (ablation `deal_arb`).

## 3 · The model

```
per company: last 20 days of price indicators + news features
        │
      LSTM ──► h_i ──────────────────────────────┐
        │                                        ▼
        └──► GAT, one per edge type, summed ──► g_i ──► dense([h_i ; ELU(g_i)]) ──► P(up)
             (sector · comention · corr)
```

The dense layer sees the company's own state `h_i` beside the graph signal `g_i`. Remove the
GAT and the same class is **Baseline 6** (LSTM + news, no graph), trained by the same code with the
same settings and seed. The gap between the two is therefore the graph's. **Baseline 5** is Baseline
6 without the news features. All three use tuned config T1 (`results/tables/tuning_gat_lstm.md`).

In [4]:
stated = {"proposed (GAT + LSTM)": 88_385, "Baseline 6 (LSTM + news)": 35_905}
ckpt = data_file("attention/checkpoints/gat_lstm_h1_fold0.pt")
if ckpt is not None:
    try:
        import torch
        n = sum(t.numel() for t in torch.load(ckpt, map_location="cpu").values())
        print(f"Parameters in the saved proposed-model checkpoint: {n:,}")
    except ImportError:
        print("torch is not installed: showing the stated counts")
for name, n in stated.items():
    print(f"{name:26s} {n:>7,} parameters  (report Section 4)")

Parameters in the saved proposed-model checkpoint: 88,385
proposed (GAT + LSTM)       88,385 parameters  (report Section 4)
Baseline 6 (LSTM + news)    35,905 parameters  (report Section 4)


## 4 · The result: price → + news → + graph

The headline metric is **within-day AUC**, the AUC among one day's companies averaged over days. The
research question is cross-sectional: does news about one firm tell us about the firms linked to it?
Pooled **fold AUC**, which ranks all of a fold's rows together, was set aside as the headline, because
it also rewards calling whole days up or down. That is market timing, which the graph has nothing to
do with. It is kept below as a diagnostic.

Each cell is the mean over the 5 folds; seeds 42, 43, 44, with seed 42 the official run. Read each
step against the seed noise printed under its table:
- **1 day:** the graph step (+0.0052) is below the proposed model's seed range (0.0058), and positive
  in only 9 of 15 fold-seeds. The news step between
  the baselines (B6 − B5, no graph: +0.0016, 1 of 3 seeds) is within noise.
- **5 days:** Baseline 5 ranges from 0.481 to 0.504 across seeds. The 5-day news step between
  the baselines (B6 − B5: +0.009) is therefore mostly Baseline 5's seed spread, and the graph step (−0.0016) is within noise.

In [5]:
A = "results/tables/ablations_3_9.md"
for h in ("1 day", "5 days"):
    display(Markdown(f"**{h}**"), md_table(A, f"## {h}", "### Ladder"), md_bullets(A, f"## {h}", "### Ladder"))
    display(md_table(A, f"## {h}", "Each step by fold"))
display(Markdown("**Fold AUC (diagnostic, includes market timing)**"),
        md_table(A, "## Fold AUC (diagnostic)").head(3),
        Markdown("At 1 day the two metrics rank the models in opposite orders: fold AUC puts Baseline 5 "
                 "above the proposed model (0.505 against 0.501), within-day AUC puts it below (0.500 against "
                 "0.507). The difference is market timing, which fold AUC rewards and within-day AUC removes."))

**1 day**

,model,seed 42,seed 43,seed 44,mean (range)
0,B5 LSTM price,0.499,0.502,0.498,0.500 (0.498 to 0.502)
1,B6 LSTM + news,0.505,0.501,0.498,0.501 (0.498 to 0.505)
2,proposed,0.509,0.507,0.504,0.507 (0.504 to 0.509)


- news step (B6 - B5): +0.0016 (-0.0008 to +0.0057; > 0 in 1/3 seeds, 9/15 fold-seeds)
- graph step (proposed - B6): +0.0052 (+0.0043 to +0.0059; > 0 in 3/3 seeds, 9/15 fold-seeds)
- seed noise floor: the proposed model's seed range is 0.0058
- ladder monotone (both steps > 0) in 3/5 folds at seed 42, 1/5 folds at seed 43, 1/5 folds at seed 44

,step,fold 0,fold 1,fold 2,fold 3,fold 4
0,news step (B6 - B5),0.0052,0.0053,0.0026,0.0031,-0.0082
1,graph step (proposed - B6),-0.0020,-0.0045,0.0074,0.0207,0.0043


**5 days**

,model,seed 42,seed 43,seed 44,mean (range)
0,B5 LSTM price,0.481,0.504,0.489,0.491 (0.481 to 0.504)
1,B6 LSTM + news,0.507,0.500,0.494,0.500 (0.494 to 0.507)
2,proposed,0.499,0.500,0.497,0.499 (0.497 to 0.500)


- news step (B6 - B5): +0.0090 (-0.0047 to +0.0262; > 0 in 2/3 seeds, 10/15 fold-seeds)
- graph step (proposed - B6): -0.0016 (-0.0081 to +0.0031; > 0 in 2/3 seeds, 7/15 fold-seeds)
- seed noise floor: the proposed model's seed range is 0.0028
- ladder monotone (both steps > 0) in 1/5 folds at seed 42, 1/5 folds at seed 43, 1/5 folds at seed 44

,step,fold 0,fold 1,fold 2,fold 3,fold 4
0,news step (B6 - B5),0.0206,0.0054,0.0306,-0.0062,-0.0057
1,graph step (proposed - B6),0.0011,0.0016,-0.0185,0.0070,0.0011


**Fold AUC (diagnostic, includes market timing)**

,variant,1d,5d
0,B5 LSTM price,0.505 (0.503 to 0.506),0.496 (0.494 to 0.502)
1,B6 LSTM + news,0.499 (0.496 to 0.503),0.506 (0.505 to 0.508)
2,proposed,0.501 (0.495 to 0.509),0.508 (0.501 to 0.517)


At 1 day the two metrics rank the models in opposite orders: fold AUC puts Baseline 5 above the proposed model (0.505 against 0.501), within-day AUC puts it below (0.500 against 0.507). The difference is market timing, which fold AUC rewards and within-day AUC removes.

## 5 · The ablation: news features × co-mention edges

A 2×2 inside the graph model. `sector` and `corr` edges are in all four cells, so the columns test
the co-mention edges, the edge type built from news.
Each news effect here is the contrast inside the graph model (news features on vs off, same edges),
not the B6 − B5 step of Section 4.
- **1 day:** news features add about +0.004 to +0.005 whether or not co-mention edges are present,
  at 3/3 seeds. Co-mention edges add about nothing whether or not news features are present (−0.0009
  and −0.0001, within noise).
- **5 days:** news features add +0.0035 inside the graph model, at 2/3 seeds: within noise. Co-mention edges cost about 0.005,
  at 0/3 seeds positive.

In [6]:
for h in ("1 day", "5 days"):
    display(Markdown(f"**{h}**"), md_table(A, f"## {h}", "### News: signal vs structure"),
            md_bullets(A, f"## {h}", "### News: signal vs structure"))

**1 day**

,,comention edges,no comention edges
0,news features,0.507 (0.504 to 0.509) (proposed),0.507 (0.503 to 0.511)
1,no news features,0.502 (0.498 to 0.506),0.502 (0.501 to 0.504)


- signal only vs structure only: +0.0050 (+0.0018 to +0.0081; > 0 in 3/3 seeds, 10/15 fold-seeds)
- news features, with comention present: +0.0041 (+0.0007 to +0.0064; > 0 in 3/3 seeds, 9/15 fold-seeds)
- news features, without comention: +0.0049 (+0.0020 to +0.0095; > 0 in 3/3 seeds, 11/15 fold-seeds)
- comention edges, with news features: -0.0009 (-0.0017 to +0.0001; > 0 in 1/3 seeds, 7/15 fold-seeds)
- comention edges, without news features: -0.0001 (-0.0031 to +0.0015; > 0 in 2/3 seeds, 9/15 fold-seeds)

**5 days**

,,comention edges,no comention edges
0,news features,0.499 (0.497 to 0.500) (proposed),0.504 (0.501 to 0.505)
1,no news features,0.495 (0.490 to 0.505),0.500 (0.495 to 0.508)


- signal only vs structure only: +0.0083 (-0.0012 to +0.0151; > 0 in 2/3 seeds, 10/15 fold-seeds)
- news features, with comention present: +0.0035 (-0.0055 to +0.0090; > 0 in 2/3 seeds, 7/15 fold-seeds)
- news features, without comention: +0.0035 (-0.0039 to +0.0089; > 0 in 2/3 seeds, 9/15 fold-seeds)
- comention edges, with news features: -0.0048 (-0.0061 to -0.0039; > 0 in 0/3 seeds, 4/15 fold-seeds)
- comention edges, without news features: -0.0047 (-0.0062 to -0.0027; > 0 in 0/3 seeds, 5/15 fold-seeds)

## 6 · Why the graph adds so little

- **Attention is near-uniform.** Entropy is 0.99–1.00 of its maximum for every edge type, and the
  top neighbour gets 1.03–1.08 × the uniform share. The GAT averages its neighbours rather than
  selecting among them.
- **The contribution is mostly a day-level offset.** The graph moves the logit by a median 0.145,
  but 58% (1 day) to 67% (5 days) of that variance is common to every ticker on the day. Within-day
  AUC correctly ignores it, since it moves every company alike.

In [7]:
S = "results/attention/summary.md"
display(md_table(S, "## Attention concentration")[["horizon", "edge_type", "median_k", "mean_entropy",
                                                   "median_top_share", "median_uniform", "top / uniform (median)"]])
c = md_table(S, "## Graph contribution to each prediction").set_index("")
display(c.loc[["abs(graph logit): median", "day-level share of graph-logit variance", "within-day part: median abs",
               "within-day AUC, graph on", "within-day AUC, graph off"]])

,horizon,edge_type,median_k,mean_entropy,median_top_share,median_uniform,top / uniform (median)
0,1,comention,10.000,0.997,0.106,0.100,1.043
1,1,corr,11.000,0.992,0.103,0.091,1.079
2,1,sector,6.000,0.999,0.168,0.167,1.030
3,5,comention,10.000,1.000,0.104,0.100,1.036
4,5,corr,11.000,0.999,0.096,0.091,1.056
5,5,sector,6.000,0.999,0.170,0.167,1.036


,1d,5d
,,
abs(graph logit): median,0.145,0.152
day-level share of graph-logit variance,0.58,0.67
within-day part: median abs,0.0387,0.0274
"within-day AUC, graph on",0.5094,0.4992
"within-day AUC, graph off",0.5052,0.4968


## 7 · One worked prediction: SGEN, 17 May 2023, 1 day

The candidates are ranked by how much the graph changed the day's ranking
(`results/attention/top_contributions.csv`). The top two are GOOG and GOOGL, one company counted twice,
so the demo takes the next one. SGEN had no news of its own that day. Its attention is near-uniform
over its `sector` and `corr` neighbours. Read this as what the model attended to, not as a mechanism
the project found. Picked for its size, it shows how propagation works, not how often it helps.

The graph moves P(up) from 0.490 to 0.914, and removing the co-mention edges alone takes +1.975 off
the +2.405 graph logit. SGEN has exactly one co-mention neighbour, VRTX, so the softmax gives that
edge the full weight regardless of its Jaccard of 0.003. A sole weak neighbour carries as much as a
sole strong one. That is a property of the normalisation rather than evidence of a relationship, and
plausibly one reason the graph does not help on average.

In [8]:
cand = pd.read_csv(ATTN / "top_contributions.csv")
row = cand[(cand["horizon"] == 1) & (cand["date"] == "2023-05-17") & (cand["ticker"] == "SGEN")].iloc[0]
types = ["comention", "corr", "sector"]
display(pd.Series({
    "P(up)": f"{row['p_up']:.3f}", "P(up), graph off": f"{row['p_up_no_graph']:.3f}",
    "realised": "up" if row["y_true"] == 1 else "down", "own news that day": "yes" if row["has_news"] else "no",
    "graph logit (all edges)": f"{row['graph_logit']:+.3f}",
    "  within-day part": f"{row['graph_logit_within_day']:+.3f}",
    **{f"  via {t} ({int(row[f'in_degree_{t}'])} edges)": f"{row[f'graph_logit_{t}']:+.3f}" for t in types},
}).to_frame("SGEN, 17 May 2023"))

att = data_file("attention/gat_lstm_edges.parquet")
if att is not None:
    inc = pd.read_parquet(att, filters=[("dst", "==", "SGEN"), ("horizon", "==", 1)])
    inc = inc[pd.to_datetime(inc["date"]) == "2023-05-17"]
    title = "All incoming edges"
else:
    inc = pd.read_csv(REPO / "results/interpretability/sgen_2023-05-17_h1.csv")
    inc = inc.drop_duplicates(["edge_type", "src"])
    title = "Top 3 incoming edges per type (results/interpretability/sgen_2023-05-17_h1.csv)"
inc = inc.assign(uniform=1 / inc["edge_type"].map(lambda t: row[f"in_degree_{t}"]))
inc = inc.sort_values(["edge_type", "alpha_mean"], ascending=[True, False])
print(title + ": attention (mean of 4 heads) against the uniform share 1/k")
inc[["edge_type", "src", "weight", "alpha_mean", "uniform"]].reset_index(drop=True).round(3)

,"SGEN, 17 May 2023"
P(up),0.914
"P(up), graph off",0.490
realised,up
own news that day,no
graph logit (all edges),+2.405
within-day part,+2.717
via comention (1 edges),+1.975
via corr (8 edges),+0.198
via sector (6 edges),+0.168


All incoming edges: attention (mean of 4 heads) against the uniform share 1/k


,edge_type,src,weight,alpha_mean,uniform
0,comention,VRTX,0.003,1.000,1.0
1,corr,DXCM,0.086,0.162,0.125
2,corr,VRTX,0.278,0.141,0.125
3,corr,CRWD,0.104,0.131,0.125
4,corr,AMGN,0.111,0.130,0.125
5,corr,BIIB,0.140,0.114,0.125
6,corr,ENPH,0.083,0.112,0.125
7,corr,IDXX,0.095,0.106,0.125
8,corr,REGN,0.158,0.104,0.125
9,sector,AMGN,1.000,0.183,0.167


## 8 · Conclusion

- **News adds a small, consistent cross-sectional signal** inside the model: +0.004 to +0.005
  within-day AUC at 1 day, positive at 3/3 seeds. Without the graph (Baseline 6 − Baseline 5) the
  same step is +0.002 at 1/3 seeds, which is within noise.
- **Graph propagation adds nothing detectable.** The 1-day graph step (+0.005) is within the proposed
  model's seed range (0.006) and positive in 9 of 15 fold-seeds. Fold 3 carries the mean (+0.021); the
  other folds range from −0.0045 to +0.0074. At 5 days the step is −0.002.
- **The hypothesis is not supported, and we report that.** The pre-registered horizon pattern (graph
  helps at 5 days, not at 1) also did not hold (`results/tables/ladder_3_8.md`).